# Stenosis — ARCADE-ONLY control (honest split)

Purpose: isolate whether **Danilov helps or hurts**. Same recipe as the ARCADE+Danilov run
(yolo11s @ 768, honest patient/per-image split) but **Danilov is force-dropped** — trains on
ARCADE stenosis alone. Compare its honest F1/mAP50 to the mixed run's (mAP50 0.10 / F1 0.19).

### Kaggle setup
1. **GPU ON, Internet ON.**
2. **Add Input:** attach **ONLY the ARCADE** dataset (the one with `stenosis/*/annotations/*.json`). Do NOT attach Danilov.
3. `DRY_RUN` is False. **Save Version -> Save & Run All (Commit)** so output registers.
4. Pull `runs/stenosis/arcade_yolo11s_768_e150/base/weights/best.pt` + `results.csv`.


## 1 · GPU + clone + install

In [ ]:
import os, sys, torch
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU.'
print('CUDA:', torch.cuda.get_device_name(0))

REPO_SLUG    = 'jugalmodi0111/interventional-imaging-pipeline'
GITHUB_TOKEN = ''                              # '' if repo is PUBLIC, else paste a PAT
REPO         = '/kaggle/tmp/interventional-imaging-pipeline'  # /kaggle/tmp NOT saved to output -> repo+9k PNGs stay out of download
if not os.path.exists(REPO):
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone -q https://{auth}github.com/{REPO_SLUG}.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install 'ultralytics>=8.2' pycocotools opencv-python pyyaml
from src import env
E = env.setup()
torch.backends.cudnn.benchmark = True

## 2 · Auto-detect + link the attached datasets
No paths to edit — finds ARCADE and any *danilov* folder at any depth under `/kaggle/input`.

In [ ]:
import glob, os
# ARCADE-only control: detect ARCADE, force Danilov OFF.
ah = glob.glob('/kaggle/input/**/stenosis/*/annotations/*.json', recursive=True)
assert ah, 'No ARCADE stenosis json under /kaggle/input — + Add Input the ARCADE dataset.'
ARCADE_INPUT = ah[0].split('/stenosis/')[0]
DANILOV_INPUT = ''                     # force ARCADE-only (do not attach/convert Danilov)
print('ARCADE_INPUT =', ARCADE_INPUT); print('DANILOV_INPUT = (forced OFF - ARCADE only)')
os.makedirs('data/raw', exist_ok=True)
os.system(f'ln -sfn "{ARCADE_INPUT}" data/raw/arcade')
assert glob.glob('data/raw/arcade/stenosis/**/*.json', recursive=True), 'ARCADE symlink wrong'


## 3 · Standardize ARCADE (+ Danilov) → YOLO

In [ ]:
import yaml, glob
cfg = yaml.safe_load(open('configs/stenosis_yolo.yaml'))
if not DANILOV_INPUT:
    cfg['datasets'].pop('danilov', None)                  # ARCADE-only if Danilov not attached
from src.data_prep import danilov_to_yolo
danilov_to_yolo.main(cfg)                                  # prints 'arcade_stenosis: N' and 'danilov: N'
ntr = len(glob.glob('data/processed/stenosis/images/train/*'))
nva = len(glob.glob('data/processed/stenosis/images/val/*'))
print(f'train imgs: {ntr} | val imgs: {nva}')
assert ntr and nva, 'conversion produced no images — check the printed per-dataset counts above'

## 4 · Train YOLO11s @ 768
Leave `DRY_RUN = True` for a 3-epoch wiring check. Set `False` for the real 150-epoch run (target **F1 > 0.57**). OOM auto-falls back to batch 8.

In [ ]:
DRY_RUN = False         # honest re-run: patient-grouped split (io_utils.group_key). Run All.

if DRY_RUN:
    cfg['train']['epochs'] = 3
    cfg['ssl'] = {'pseudo_label': False}                  # skip SSL on the fast pass
from src.train.train_detector import train, train_kwargs, run_tag
TAG = run_tag(cfg)
print('run:', TAG, '| kwargs ->', train_kwargs(cfg))
try:
    best = train(cfg, project=f"{E['runs']}/stenosis/{TAG}")
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        torch.cuda.empty_cache(); cfg['train']['batch'] = 8
        print('CUDA OOM at batch 16 -> retrying at batch 8'); best = train(cfg, project=f"{E['runs']}/stenosis/{TAG}")
    else:
        raise
print('best weights ->', best)

## 5 · Collect outputs for download
Copies `best.pt` + `results.csv` to `/kaggle/working` and zips the full run. Grab them from the **Output** panel (right).

In [ ]:
import shutil, os
run_dir = os.path.dirname(os.path.dirname(best))          # .../<TAG>/base
for rel in ('weights/best.pt', 'results.csv'):
    src = os.path.join(run_dir, rel)
    if os.path.exists(src):
        dst = f'/kaggle/working/{os.path.basename(rel)}'
        shutil.copy(src, dst); print('download ->', dst)
shutil.make_archive('/kaggle/working/stenosis_run', 'zip', run_dir)
print('full run zip -> /kaggle/working/stenosis_run.zip')
print('\nNext (on your Mac): python -m src.export.yolo_to_coreml --weights best.pt')